# Paso 1: Creación de base de datos y esquemas

## Librerias

In [1]:
# Generales
import pandas as pd
import numpy as np
import os 

# Ocultar warnings
import warnings
warnings.filterwarnings('ignore')

# Aumentar número de columnas que se pueden ver
pd.options.display.max_columns = None
# En los dataframes, mostrar los float con dos decimales
pd.options.display.float_format = '{:,.10f}'.format
# Cada columna será tan grande como sea necesario para mostrar todo su contenido
pd.set_option('display.max_colwidth', 0)

In [2]:
# Cambiar directorio para importar modulos y datos fácilmente
os.chdir('../')
os.getcwd()

'c:\\Users\\nrivera\\OneDrive - PROCOLOMBIA\\Documentos\\029-App-Segmentacion-Analitica\\app-segmentacion-exportaciones'

In [3]:
# Modulo de Snowflake Analítica
import src.snowflake_analitica as snowflake_analitica

2025-04-14 14:24:40.604 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-14 14:24:40.605 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2025-04-14 14:24:40.606 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-14 14:24:40.606 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-14 14:24:40.608 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-14 14:24:40.608 WARNING streamlit.runtime.scriptrunner_utils.script_run_c

### Snowflake

In [4]:
# Creación de sesión
json_path = './.streamlit/snowflake_credentials.json'
sesion_activa, conexion_activa = snowflake_analitica.create_session_from_json(json_file_path = json_path)
sesion_activa

## 1. Crear base de datos

In [ ]:
# Query para crear la base de datos
sql_database = """
CREATE OR REPLACE DATABASE APP_SEGMENTACION_EXPORTACIONES;
"""
# Ejecutar
sesion_activa.sql(sql_database).collect()

In [ ]:
# Identificar ubicación actual
snowflake_analitica.get_session_info(sesion_activa)

## 2. Crear esquemas

In [ ]:
# Crear esquema seguimiento:
sql_schema_seguimiento = """
CREATE OR REPLACE SCHEMA SEGUIMIENTO;
"""
# Ejecutar
sesion_activa.sql(sql_schema_seguimiento).collect()

In [ ]:
# Crear esquema para la página de segmentación:
sql_schema_segmentacion = """
CREATE OR REPLACE SCHEMA SEGMENTACION;
"""
# Ejecutar
sesion_activa.sql(sql_schema_segmentacion).collect()

## 3. Crear tablas

### Seguimiento de eventos de la aplicación

In [5]:
# Crear tabla para seguimiento de eventos
sql_seguimiento_tabla = """
CREATE OR REPLACE TABLE APP_SEGMENTACION_EXPORTACIONES.SEGUIMIENTO.EVENTOS (
	TIPO_EVENTO VARCHAR(16777216),
    PAGINA VARCHAR(16777216),
	DETALLE_EVENTO VARCHAR(16777216),
	FILTROS VARCHAR(16777216),
	FECHA_HORA TIMESTAMP_NTZ(9)
);
"""
# Ejecutar
sesion_activa.sql(sql_seguimiento_tabla).collect()

[Row(status='Table EVENTOS successfully created.')]

### Seguimiento de cargues de información

In [ ]:
# Usar el esquema de auditoria
snowflake_analitica.update_session_params(sesion_activa, database='APP_SEGMENTACION_EXPORTACIONES', schema='SEGUIMIENTO')

In [ ]:
# Crear tabla para generar IDs
sesion_activa.sql("CREATE TABLE IF NOT EXISTS ID_GENERATOR (ID INTEGER PRIMARY KEY);").collect()
sesion_activa.sql("INSERT INTO ID_GENERATOR (ID) VALUES (0);").collect()

# Crear procedimiento almacenado para obtener el próximo ID
procedimiento = """
CREATE OR REPLACE PROCEDURE GET_NEXT_ID()
RETURNS INTEGER
LANGUAGE SQL
AS $$
DECLARE
    NEXT_ID INTEGER;
BEGIN
    -- Incrementar el ID
    UPDATE ID_GENERATOR
    SET ID = ID + 1;

    -- Obtener el próximo ID
    SELECT ID INTO NEXT_ID
    FROM ID_GENERATOR;

    RETURN NEXT_ID;
END;
$$
"""

In [ ]:
# Ejecutar el procedimiento en Snowflake
sesion_activa.sql(procedimiento).collect()

In [ ]:
# Crear tabla de auditoria de cargue de datos:
sql_tabla_auditoria_cargue = """
CREATE TABLE AUDITORIA_CARGUES (
    ID_AUDITORIA        INTEGER,                                -- Identificador único
    NOMBRE_ESQUEMA_DESTINO VARCHAR(255) NOT NULL,               -- Esquema de destino del cargue
    NOMBRE_TABLA        VARCHAR(255) NOT NULL,                  -- Nombre de la tabla de destino
    FECHA_CARGUE        TIMESTAMP DEFAULT CURRENT_TIMESTAMP,    -- Fecha y hora del cargue
    NUMERO_REGISTROS    INTEGER,                                -- Número de registros cargados
    MENSAJE             VARCHAR(512) NOT NULL                   -- Mensaje de resultado del cargue a Snowflake
);
"""
# Ejecutar
sesion_activa.sql(sql_tabla_auditoria_cargue).collect()

## 4. Cerrar conexión

In [ ]:
sesion_activa.close()